# Lekcija 13 - Pomnilnik agenta s Cognee grafi znanja


## Namestitev

Ta zvezek prikazuje, kako zgraditi inteligentnega **pomočnika za programiranje** s trajnim spominom z uporabo znanstvenih grafov [**Cognee**](https://www.cognee.ai/) in **Microsoft Agent Framework** (MAF).

Cognee pretvori nestrukturirano besedilo v strukturiran, poizvedljiv znanstveni graf, podprt z vektorskimi u embeddedi — kar vašemu agentu daje bogat, z odnosi ozaveščen dolgoročni spomin.

### Kaj se boste naučili
1. **Gradnja znanstvenih grafov**: Pretvorite profile razvijalcev in najboljše prakse v strukturirano, poizvedljivo znanje.
2. **Integracija Cognee z MAF**: Uporabite funkcije `@tool`, da agent MAF poizveduje po Cogneejevem znanstvenem grafu.
3. **Pogovori, ki upoštevajo sejo**: Ohranjajte kontekst skozi več vprašanj iste seje.
4. **Dolgoročni spomin**: Ohranjajte pomembno znanje med sejami in ga pridobite v novih pogovorih.

### Predpogoji
- Python 3.9+
- Lokalen zagon Redis (`docker run -d -p 6379:6379 redis`) za upravljanje sej
- Ključ API za LLM (npr. OpenAI) — nastavite `LLM_API_KEY` v `.env`
- `CACHING=true` v `.env` (zahtevano za Cognee seje)
- Projekt Azure AI Foundry z nameščenim modelom za klepet
- `AZURE_AI_PROJECT_ENDPOINT` in `AZURE_AI_MODEL_DEPLOYMENT_NAME` v `.env`
- Prijava v Azure CLI (`az login`)


In [ ]:
%pip install agent-framework azure-ai-projects azure-identity "cognee[redis]==0.4.0" -q

In [ ]:
import os
from pathlib import Path
from typing import Annotated

from dotenv import load_dotenv

load_dotenv()

os.environ["LLM_API_KEY"] = os.getenv("LLM_API_KEY", "")
os.environ["CACHING"] = os.getenv("CACHING", "true")

import cognee
from cognee.modules.search.types import SearchType

from agent_framework import tool
from agent_framework.azure import AzureAIProjectAgentProvider
from azure.identity import AzureCliCredential

print(f"Cognee version: {cognee.__version__}")
print(f"CACHING: {os.environ.get('CACHING')}")

In [ ]:
provider = AzureAIProjectAgentProvider(credential=AzureCliCredential())

print("✅ AzureAIProjectAgentProvider created")

## Vrste agentovega spomina

Ta zvezek raziskuje iste tri vrste spomina kot glavni zvezek Lekcije 13, vendar uporablja Cognee kot hranilnik dolgoročnega spomina:

| Vrsta spomina | Mehanizem | Trajanje |
|---|---|---|
| **Delovni** | `agent.create_session()` (MAF) | En pogovor |
| **Kratkotrajni** | Cognee predpomnilnik sej (Redis) | Ena seja |
| **Dolgoročni** | Cognee znanstveni graf + vektorji | Trajno |

### Arhitektura spomina Cognee
```
┌──────────────────────────┐
│      Raw Data            │  (developer profiles, docs, conversations)
└───────────┬──────────────┘
            │  cognee.add() + cognee.cognify()
            ▼
┌──────────────────────────────────────────┐
│  Knowledge Graph + Vector Embeddings     │
└───────────┬──────────────────────────────┘
            │  cognee.search()
            ▼
┌──────────────────┐       ┌────────────────┐
│  MAF Agent       │──────▶│  @tool funcs   │
│  (AgentSession)  │       │  wrapping       │
│                  │       │  cognee.search  │
└──────────────────┘       └────────────────┘
```


## Priprava Cognee shrambe


In [ ]:
DATA_ROOT = Path('.data_storage').resolve()
SYSTEM_ROOT = Path('.cognee_system').resolve()

DATA_ROOT.mkdir(parents=True, exist_ok=True)
SYSTEM_ROOT.mkdir(parents=True, exist_ok=True)

cognee.config.data_root_directory(str(DATA_ROOT))
cognee.config.system_root_directory(str(SYSTEM_ROOT))

await cognee.prune.prune_data()
await cognee.prune.prune_system(metadata=True)
print("✅ Cognee storage configured and reset")

## Del 1 — Izgradnja baze znanja

Vnašamo tri vrste podatkov za ustvarjanje obsežne baze znanja za našega pomočnika pri programiranju:

1. **Profil razvijalca** — osebna strokovnost in tehnična podlaga  
2. **Najboljše prakse Python** — Zen Pythona s praktičnimi smernicami  
3. **Zgodovinski pogovori** — pretekle seje vprašanj in odgovorov med razvijalci in AI pomočniki


In [ ]:
developer_intro = (
    "Hi, I'm an AI/Backend engineer. "
    "I build FastAPI services with Pydantic, heavy asyncio/aiohttp pipelines, "
    "and production testing via pytest-asyncio. "
    "I've shipped low-latency APIs on AWS, Azure, and GoogleCloud."
)

python_zen_principles = """
# The Zen of Python: Practical Guide

## Key Principles With Guidance

### 1. Beautiful is better than ugly
Prefer descriptive names, clear structure, and consistent formatting.

### 2. Explicit is better than implicit
Be clear about behavior, imports, and types.

### 3. Simple is better than complex
Choose straightforward solutions first.

### 4. Flat is better than nested
Use early returns to reduce indentation.

## Modern Python Tie-ins
- Type hints reinforce explicitness
- Context managers enforce safe resource handling
- Dataclasses improve readability for data containers
"""

human_agent_conversations = """
"conversations": [
    {
      "topic": "async/await patterns",
      "user_query": "I'm building a web scraper that needs to handle thousands of URLs concurrently. What's the best way to structure this with asyncio?",
      "assistant_response": "Use asyncio with aiohttp, a semaphore to cap concurrency, TCPConnector for connection pooling, and context managers for session lifecycle."
    },
    {
      "topic": "dataclass vs pydantic",
      "user_query": "When should I use dataclasses vs Pydantic models?",
      "assistant_response": "For API input/output, prefer Pydantic: runtime validation, type coercion, JSON serialization. Integrates cleanly with FastAPI."
    },
    {
      "topic": "testing patterns",
      "user_query": "What's the best approach for pytest with async functions?",
      "assistant_response": "Use pytest-asyncio, async fixtures, and an isolated test database or mocks to reliably test async code."
    },
    {
      "topic": "error handling and logging",
      "user_query": "What's the best approach for production-ready error management?",
      "assistant_response": "Centralized error handling with custom exceptions, structured logging, and FastAPI middleware."
    }
  ]
"""

print("✅ Data sources prepared")

In [ ]:
await cognee.add(developer_intro, node_set=["developer_data"])
await cognee.add(human_agent_conversations, node_set=["developer_data"])
await cognee.add(python_zen_principles, node_set=["principles_data"])

await cognee.cognify()
print("✅ Knowledge graph built")

## Vizualizirajte znanstveni graf

Cognee lahko ustvari interaktivno HTML vizualizacijo entitet in odnosov, ki jih je izvlekel.


In [ ]:
from cognee import visualize_graph

await visualize_graph('./cognee_graph.html')
print("📊 Graph saved to cognee_graph.html — open it in a browser to explore.")

## Obogati pomnjenje z Memify

`memify()` analizira graf znanja in generira inteligentna pravila — identificira vzorce, najboljše prakse in odnose med pojmi.


In [ ]:
await cognee.memify()
print("✅ Memory enriched with memify")

## Del 2 — MAF agent s Cognee orodji

Zdaj ustvarimo MAF agenta, ki lahko poizveduje po Cogneejevem znanstvenem grafu prek funkcij `@tool`. To agentu omogoča, da izkoristi polno moč semantičnega iskanja, ki pozna graf, hkrati pa ohranja kontekst pogovora skozi seje.


In [ ]:
@tool(approval_mode="never_require")
async def search_knowledge(
    query: Annotated[str, "Natural-language question to search the knowledge graph"],
) -> str:
    """Search the Cognee knowledge graph for relevant developer knowledge, best practices, and past conversations."""
    results = await cognee.search(
        query_text=query,
        query_type=SearchType.GRAPH_COMPLETION,
    )
    if not results:
        return "No relevant knowledge found."
    return str(results)


@tool(approval_mode="never_require")
async def search_principles(
    query: Annotated[str, "Question about Python principles or best practices"],
) -> str:
    """Search only the Python principles subset of the knowledge graph."""
    from cognee.modules.engine.models.node_set import NodeSet
    results = await cognee.search(
        query_text=query,
        query_type=SearchType.GRAPH_COMPLETION,
        node_type=NodeSet,
        node_name=["principles_data"],
    )
    if not results:
        return "No relevant principles found."
    return str(results)


print("✅ Cognee tools defined: search_knowledge, search_principles")

In [ ]:
coding_agent = await provider.create_agent(
    name="CodingAssistant",
    instructions=(
        "You are an expert coding assistant with access to a knowledge graph "
        "containing developer profiles, Python best practices, and past conversations.\n\n"
        "WORKFLOW:\n"
        "1. Use search_knowledge() to find relevant information from the full knowledge graph.\n"
        "2. Use search_principles() when the question is specifically about Python best practices.\n"
        "3. Combine retrieved knowledge with your own expertise to give comprehensive answers.\n"
        "4. Reference the developer's known tech stack (FastAPI, asyncio, Pydantic) when relevant."
    ),
)

print("✅ CodingAssistant agent created")

## Delovni spomin s sejami

`AgentSession` (ustvarjen prek `agent.create_session()`) zagotavlja delovni spomin znotraj seje. Agent se lahko sklicuje na prejšnja sporočila, hkrati pa poizveduje po dolgoročnem grafu znanja Cognee.


In [ ]:
session = coding_agent.create_session()

response = await coding_agent.run(
    "How does my AsyncWebScraper implementation align with Python's design principles?",
    session=session,
)
print("🤖 Agent:", response)

In [ ]:
response = await coding_agent.run(
    "Based on what you just said, when should I pick dataclasses versus Pydantic for this work?",
    session=session,
)
print("🤖 Agent:", response)
print("\n💡 The agent combined working memory (previous answer) with Cognee's knowledge graph.")

## Nova seja — dolgoročni spomin ostane

Začetek nove seje počisti delovni spomin, vendar je znanje iz Cogneevega grafu še vedno na voljo. Agent lahko v popolnoma novem pogovoru pridobi isto dolgoročno znanje.


In [ ]:
session_2 = coding_agent.create_session()

response = await coding_agent.run(
    "What logging guidance should I follow for incident reviews?",
    session=session_2,
)
print("🤖 Agent:", response)
print("\n💡 New session, but the agent still has access to the full Cognee knowledge graph.")

In [ ]:
response = await coding_agent.run(
    "How should variables be named according to Python best practices?",
    session=session_2,
)
print("🤖 Agent:", response)

## Povzetek

V tem zapisku ste zgradili kodirnega asistenta, ki združuje **MAF-ov delovni pomnilnik** (`agent.create_session()`) s **Cogneejevim dolgoročnim grafom znanja**.

### Kaj ste se naučili
1. **Gradnja grafa znanja**: Cognee obdela nestrukturirano besedilo in zgradi graf + vektorski pomnilnik.
2. **Obogatitev grafa z memify**: Izpeljani podatki in bogatejši odnosi na vrhu obstoječega grafa.
3. **Integracija MAF + Cognee**: `@tool` funkcije omogočajo MAF agentu, da naravno poizveduje po Cogneejevem grafu.
4. **Delovni pomnilnik + dolgoročni pomnilnik**: `AgentSession` (prek `agent.create_session()`) nudi kontekst seje, medtem ko Cognee zagotavlja trajno znanje.
5. **Filtrirano iskanje z NodeSets**: Ciljanje na specifične podmnožice grafa znanja (npr. samo načela).

### Ključni poudarki
- **Cognee** spremeni surovo besedilo v strukturiran, odnosno-zavedajoč se pomnilnik — močnejši od enostavne vektorske baze.
- **`@tool` funkcije** jasno povezujejo MAF agente z zunanjimi sistemi znanja.
- **`AgentSession`** (prek `agent.create_session()`) ohranja kontekst posameznega pogovora ločeno od dolgoročnega znanja.
- Enak graf znanja služi več sej in agentom.

### Praktične uporabe
- **Razvojni pomočniki**: Pregled kode, analiza incidentov, asistenti za arhitekturo
- **Pomoč uporabnikom**: Agent za podporo prek dokumentacije, pogostih vprašanj in CRM zapiskov
- **Notranji strokovni pomočniki**: Asistenti za politike, pravne ali varnostne zadeve, ki razmišljajo prek smernic
- **Združena podatkovna plast**: Združevanje strukturiranih in nestrukturiranih podatkov v en pobarvljiv graf

### Naslednji koraki
- Eksperimentirajte z časovno zavednostjo v Cogneeju
- Določite OWL ontologijo za kakovost grafa specifično za domeno
- Dodajte povratne zanke uporabnikov za izboljšanje pridobivanja skozi čas
- Razširite na večagentne sisteme, ki si delijo isti Cognee pomnilniški sloj


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**Omejitev odgovornosti**:
Ta dokument je bil preveden z uporabo AI storitve za prevajanje [Co-op Translator](https://github.com/Azure/co-op-translator). Čeprav si prizadevamo za natančnost, vas opozarjamo, da avtomatizirani prevodi lahko vsebujejo napake ali netočnosti. Izvirni dokument v njegovem izvornem jeziku velja za avtoritativni vir. Za pomembne informacije priporočamo strokovni človeški prevod. Ne odgovarjamo za morebitne nesporazume ali napačne interpretacije, ki izhajajo iz uporabe tega prevoda.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
